# 05 — RAG: Retrieving Similar Historical Complaints

**Purpose:** given a new complaint, retrieve similar historical complaints along with how each was resolved. This gives a fast, evidence-based answer to "what usually happens with complaints like this one" without training a dedicated classifier — and it's the retrieval foundation Notebook 06 builds on to generate grounded responses rather than free-form guesses.

The corpus here is many short documents — one per complaint — rather than a handful of long documents. Each "chunk" is naturally one complaint, tagged with its outcome and other structured metadata, rather than a fragment of a longer document that would need splitting.

## Loading the Cleaned Corpus from Notebook 04

Loading `data/04_complaints_model_ready.csv` instead of the raw complaints file. This is already deduplicated on narrative text — important here specifically, since duplicate narratives would make retrieval look artificially good, returning near-identical "similar" complaints that are really just copies of the same template. It also already has `Issue` consolidated (Notebook 02's two renamed-duplicate categories merged) and the 78 rows with missing `Company response to consumer` dropped.

It carries `topic_name` (Notebook 02) and the entity flags (Notebook 03) merged in as well, so those are available as retrieval metadata without re-merging them here.

## Environment Setup

Mounting Drive and installing the LangChain packages this notebook needs (`langchain-community`, `langchain-text-splitters`, plus `faiss-cpu` and `sentence-transformers` for the vector store and embedding model) — this Colab environment doesn't have them preinstalled, and recent LangChain versions split what used to be one package into several, so the exact imports below are pinned to what's actually available.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/ANLP_portfolio/")


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [5]:
# !pip list 2>/dev/null | grep -i langchain

In [9]:
!pip install -q langchain-community langchain-text-splitters faiss-cpu sentence-transformers --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 204.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

df = pd.read_csv("data/04_complaints_model_ready.csv", low_memory=False)
df = df.dropna(subset=['Consumer complaint narrative'])
print(f"Loaded from Notebook 04: {df.shape}")

Loaded from Notebook 04: (164455, 23)


## Recovering Notebook 04's Train/Test Split

Reapplying the same class merge and the same `train_test_split` call (`random_state=42`, same `stratify`) used throughout Notebook 04. This isn't for modeling here — it's so the retrieval corpus and the "held-out" complaints used in the quality check below come from the same partition Notebook 04 used, rather than an independent random split. Indexing a complaint into the retrieval corpus and then also using it as a "held-out" query later would be a quiet leakage problem — the RAG equivalent of testing on your training data.

In [5]:
TARGET_MERGE_MAP = {
    'In progress': 'Other',
    'Untimely response': 'Other',
}
y_merged = df['Company response to consumer'].replace(TARGET_MERGE_MAP)

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y_merged
)
print(f"Corpus (train) size: {len(train_idx)}, Held-out (test) size: {len(test_idx)}")

Corpus (train) size: 131564, Held-out (test) size: 32891


## Build Documents

Each complaint becomes one `Document`, built only from the **training partition** — the retrieval corpus a new complaint would actually be compared against. Metadata now includes `topic_name` and the entity flags from Notebook 04's merged dataset, not just the original CFPB fields, so retrieval results can be filtered or explained using the same signals the classification notebook found predictive.

In [6]:
corpus_df = df.iloc[train_idx]

documents = [
    Document(
        page_content=row['Consumer complaint narrative'],
        metadata={
            'product': row.get('Product'),
            'issue': row.get('Issue'),
            'topic': row.get('topic_name'),
            'has_dollar_amount': row.get('has_dollar_amount'),
            'has_company_org': row.get('has_company_org'),
            'company': row.get('Company'),
            'response': row.get('Company response to consumer'),
            'complaint_id': row.get('Complaint ID'),
        }
    )
    for _, row in corpus_df.iterrows()
]
print(f"{len(documents):,} documents in retrieval corpus")

131,564 documents in retrieval corpus


## Embed and Index

Using `sentence-transformers/all-MiniLM-L6-v2` — a small, fast sentence-embedding model well suited to short-to-medium text like complaint narratives, and a reasonable default when the goal is semantic similarity search rather than maximum embedding quality at any cost.

At 131,564 documents, embedding the full corpus in one `FAISS.from_documents()` call is a real batch job but a manageable one for this model size — proceeding directly rather than subsampling first, since the corpus is a known, fixed size (not the open-ended scale that would make a validate-on-a-sample-first approach necessary).

In [7]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# TODO: for large corpora, this should run in batches -- FAISS.from_documents on
# hundreds of thousands of docs in one call may be slow/memory-heavy. Consider
# subsampling first to validate the pipeline, then scale up.
vectorstore = FAISS.from_documents(documents, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})


/tmp/ipykernel_3132/2928453085.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Test Retrieval

A quick manual sanity check before the quantitative evaluation below: hand-writing one plausible new complaint and inspecting what comes back, to confirm the retriever returns something topically sensible before trusting it on the larger held-out sample.

In [8]:
query = "my credit card company closed my account without warning after a dispute"
results = retriever.invoke(query)

for doc in results:
    print(f"[{doc.metadata.get('response')}] {doc.page_content[:200]}...")
    print()


[Closed with explanation] I reached out to the company around XXXXXXXX XXXX XXXX and requested an account be closed since it was unauthorized and I did not apply for that credit card. On XX/XX/year> the credit card account was...

[Closed with explanation] I reached out to the company around XXXX of XXXX  and requested an account be closed since it was unauthorized and I did not apply for that credit card. On XX/XX/year> the credit card account was clos...

[Closed with explanation] I am banking with Bank Of America, I also had a credit card within them, they put a late payment on my account and then immediately closed it. My consumer laws were violated. I was not informed of any...

[Closed with non-monetary relief] My credit card was fruadlenty used and maxed out And I put in multiple disputes and identity theft complaints The bank and company failed to protect me my accounts and my money I was charged with mult...

[Closed with explanation] Bank of America gave me a credit card Clos

## Retrieval Quality Check

For a sample of complaints from the **held-out test partition** (`test_idx` — never indexed into the retrieval corpus above), checking whether the retrieved neighbors actually share the same `Issue` label as the query complaint. A simple, honest sanity check on retrieval quality before building the DSPy layer on top of it in Notebook 06. Sampling from `test_idx` specifically — rather than an arbitrary sample of the full dataset — keeps "held-out" meaning the same thing it means in Notebook 04, and makes it possible to later compare retrieval-based predictions against the classifier's predictions on the exact same rows if that's useful in Notebook 06.

In [9]:
held_out_sample = df.iloc[test_idx].sample(10, random_state=42)

match_rates = []
for i, row in held_out_sample.iterrows():
    results = retriever.invoke(row['Consumer complaint narrative'])
    match_rate = sum(r.metadata['issue'] == row['Issue'] for r in results) / len(results)
    match_rates.append(match_rate)
    print(f"{row['Issue']!r:60} -> {match_rate:.2f}")

print(f"\nMean issue-match rate across {len(held_out_sample)} held-out queries: {np.mean(match_rates):.2f}")

'Incorrect information on your report'                       -> 1.00
'Incorrect information on your report'                       -> 0.80
'Improper use of your report'                                -> 1.00
'Improper use of your report'                                -> 1.00
'Incorrect information on your report'                       -> 1.00
'Incorrect information on your report'                       -> 1.00
'Incorrect information on your report'                       -> 1.00
"Problem with a company's investigation into an existing problem" -> 0.00
'Improper use of your report'                                -> 0.80
'Incorrect information on your report'                       -> 1.00

Mean issue-match rate across 10 held-out queries: 0.86


In [10]:
print(df.iloc[train_idx]['Issue'].value_counts().get("Problem with a company's investigation into an existing problem"))

19469


In [11]:
query_row = held_out_sample[held_out_sample['Issue'] == "Problem with a company's investigation into an existing problem"].iloc[0]
results = retriever.invoke(query_row['Consumer complaint narrative'])

print(f"Query issue: {query_row['Issue']}\n")
for doc in results:
    print(f"[{doc.metadata.get('issue')}] {doc.page_content[:150]}...")
    print()

Query issue: Problem with a company's investigation into an existing problem

[Improper use of your report] XXXX Experian Information Solutions , Inc . 
XXXX XXXX XXXX XXXXXXXX XXXX XXXX Subject : Request for Deletion of Inaccurate Addresses and Name from Cr...

[Incorrect information on your report] Dear Experian XXXX  XXXX  XXXX, I am writing to formally dispute inaccurate identifying information appearing on my consumer credit report, which I re...

[Incorrect information on your report] XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX SC XXXX XXXX : XXXX XXXX : XX/XX/XXXX Equifax XXXX XXXX XXXX XXXX, GA XXXX Experian XXXX. XXXX XXXX XXXX, TX X...

[Incorrect information on your report] I am formally disputing and demanding the permanent removal of all inaccurate, obsolete, or fraudulent personal identifying information from my credit...

[Incorrect information on your report] PERSONAL INFORMATION CORRECTION & DATA MINIMIZATION DEMAND Pursuant to the Fair Credit Reporting Act ( 15 U.S.C. 168

In [12]:
similar_issues = ["Problem with a company's investigation into an existing problem",
                   "Incorrect information on your report"]
print(df.iloc[train_idx][df.iloc[train_idx]['Issue'].isin(similar_issues)]
      .groupby('Issue')['Company response to consumer'].value_counts(normalize=True))

Issue                                                            Company response to consumer   
Incorrect information on your report                             Closed with explanation            0.633259
                                                                 Closed with non-monetary relief    0.363498
                                                                 Untimely response                  0.002316
                                                                 Closed with monetary relief        0.000805
                                                                 In progress                        0.000122
Problem with a company's investigation into an existing problem  Closed with explanation            0.638040
                                                                 Closed with non-monetary relief    0.356464
                                                                 Untimely response                  0.002722
                               

## Diagnostic: The One 0.00 Match Isn't What It Looks Like

One held-out query — `Issue = "Problem with a company's investigation into an existing problem"` — got zero label matches among its 5 retrieved neighbors, pulling down the otherwise strong 0.86 mean match rate. Worth checking before treating it as a retrieval failure.

First ruled out sparsity: this `Issue` category has 19,469 examples in the training corpus, not a small class the retriever had little to work with.

Looked at what it actually retrieved instead. 4 of 5 neighbors landed on "Incorrect information on your report." Both the query and its neighbors are formal, near-boilerplate FCRA dispute letters — same structural template, same legal citations, same phrasing patterns. The embedding model is correctly picking up that these are nearly the same *kind* of document; CFPB's taxonomy just happens to split "investigation was mishandled" from "the information itself is wrong" as separate categories, a distinction that isn't showing up in narrative structure.

This tracks with something Notebook 02 already found — several LDA topics concentrated heavily in "Incorrect information on your report" regardless of topic identity. Same signature here, via a completely different method (embedding similarity instead of topic modeling).

Checked whether this label mismatch actually matters for what the retriever is meant to support — surfacing complaints with a similar likely outcome. Company response distributions across the two categories are nearly identical: 63.3% vs. 63.8% "Closed with explanation," 36.3% vs. 35.6% "Closed with non-monetary relief," and the rare classes within a fraction of a percent of each other.

So the 0.00 match isn't a retrieval failure — it's a label technicality. Three independent signals now agree that these two `Issue` categories aren't functionally distinct: topic structure (Notebook 02), embedding-space neighbors (this notebook), and resolution outcomes (this check). Which means the 0.86 mean match rate is, if anything, a slight underestimate of retrieval quality — at least one of its misses doesn't matter for the thing a user of this system actually cares about.

## Summary and Takeaway

The retrieval system works: a mean issue-match rate of 0.86 across held-out queries means the retriever is finding narratively and topically similar complaints, not just similar-length text. The one query that scored 0.00 turned out not to be a real miss — its neighbors shared the query's underlying dispute-letter template and its resolution outcome, just not its exact `Issue` label, which three separate diagnostics (topic modeling in Notebook 02, embedding similarity here, and outcome distributions) all agree is a distinction CFPB's taxonomy draws that the data itself doesn't really support. If anything, that makes 0.86 a slight underestimate of true retrieval quality.

Two design decisions carried over deliberately from Notebook 04 rather than treated as fresh choices here: the corpus is deduplicated (so "similar complaints" aren't just copies of the same template inflating apparent quality) and the retrieval corpus and held-out evaluation queries come from the same train/test partition used throughout this project (so the held-out check is a genuine test of generalization, not evaluation on documents already sitting in the index).

This gives Notebook 06 a validated retriever to build on: given a new complaint, it can reliably surface historical complaints with a similar issue and a similar likely resolution, with metadata (`topic_name`, entity flags, `Issue`) attached for filtering or explaining *why* a given result was retrieved — not just returning raw similar text.